# Скоринговая карта розничного кредитования — German Credit

**Модель вероятности дефолта (PD) на уровне заявки и перевод в баллы скоркарты**,
построенные на WOE-биннинге и логистической регрессии.

**Итоговые результаты:** Gini 0.604 · KS 0.571 · калибровка: 0.305 предсказано против 0.300
фактически (отобрано 15 из 20 признаков по критерию IV > 0.02).

## 1. Инициализация

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

from scorecard.binning import bin_categorical, bin_continuous
from scorecard.data import load_german_credit
from scorecard.inference import score_new_applicant
from scorecard.scorecard import build_scorecard, score_applicant
from scorecard.woe import apply_woe, compute_woe_iv

pd.set_option("display.max_columns", None)
SEED = 42
np.random.seed(SEED)

## 2. Данные

German Credit (UCI, id=144): 1000 заявок, 20 признаков, целевая переменная — факт дефолта
(перекодирована в 0 = good, 1 = bad). Пропусков нет. Четыре численных признака
(`installment_rate`, `residence_since`, `existing_credits`, `num_dependents`) являются
порядковыми категориями низкой кардинальности и обрабатываются как категориальные.

In [ ]:
df = load_german_credit("../data/raw/german_credit/german.data")
print(f"Размер: {df.shape}  |  Доля дефолтов: {df['target'].mean():.3f}")
df.head()

In [ ]:
categorical_cols = df.select_dtypes(include=["str"]).columns.tolist()
continuous_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
continuous_cols.remove("target")

low_card_numeric = ["num_dependents", "installment_rate", "residence_since", "existing_credits"]
categorical_cols = categorical_cols + low_card_numeric
continuous_cols = [c for c in continuous_cols if c not in low_card_numeric]

X = df.drop(columns=["target"])
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
print(f"Train: {X_train.shape}  Test: {X_test.shape}")

## 3. WOE-биннинг и Information Value

Каждый признак биннится по `X_train` (для категориальных — редкие уровни объединяются в
"other"; для непрерывных — квантильные бины с границами, расширенными до ±inf, чтобы
значения вне диапазона train корректно обрабатывались при инференсе). WOE и IV считаются
по train, затем те же бины применяются к `X_test`.

Признаки с IV < 0.02 отбрасываются как непредсказательные.

In [ ]:
iv_results, woe_tables = {}, {}
train_binned_cols, test_binned_cols = {}, {}
binning_artifacts = {}

for col in categorical_cols:
    train_b, test_b, rare = bin_categorical(X_train[col], X_test[col])
    summary, iv = compute_woe_iv(train_b, y_train)
    train_binned_cols[col], test_binned_cols[col] = train_b, test_b
    woe_tables[col], iv_results[col] = summary, iv
    binning_artifacts[col] = {
        "type": "categorical", "rare_levels": rare,
        "known_levels": X_train[col].unique().tolist(),
    }

for col in continuous_cols:
    train_b, test_b, edges = bin_continuous(X_train[col], X_test[col])
    summary, iv = compute_woe_iv(train_b, y_train)
    train_binned_cols[col], test_binned_cols[col] = train_b, test_b
    woe_tables[col], iv_results[col] = summary, iv
    binning_artifacts[col] = {"type": "continuous", "edges": edges}

iv_ranking = pd.Series(iv_results).sort_values(ascending=False)
iv_ranking

In [ ]:
plt.figure(figsize=(8, 6))
iv_ranking.sort_values().plot(kind="barh")
plt.axvline(0.02, color="red", linestyle="--", label="Порог отбора")
plt.xlabel("Information Value")
plt.title("Ранжирование признаков по IV")
plt.legend()
plt.tight_layout()
plt.savefig("../reports/figures/iv_ranking.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
keep_features = iv_ranking[iv_ranking > 0.02].index.tolist()
print(f"Отобрано {len(keep_features)} из {len(iv_ranking)} признаков")
keep_features

## 4. Модель

In [ ]:
X_train_woe = pd.DataFrame(index=X_train.index)
X_test_woe = pd.DataFrame(index=X_test.index)

for col in keep_features:
    X_train_woe[col] = apply_woe(train_binned_cols[col], woe_tables[col])
    X_test_woe[col] = apply_woe(test_binned_cols[col], woe_tables[col])

assert X_test_woe[keep_features].isna().sum().sum() == 0, "Неожиданные NaN в WOE-матрице"

model = LogisticRegression(random_state=SEED)
model.fit(X_train_woe[keep_features], y_train)

p_test = model.predict_proba(X_test_woe[keep_features])[:, 1]
auc = roc_auc_score(y_test, p_test)
gini = 2 * auc - 1
ks_stat, _ = ks_2samp(p_test[y_test == 1], p_test[y_test == 0])

print(f"AUC:   {auc:.4f}")
print(f"Gini:  {gini:.4f}")
print(f"KS:    {ks_stat:.4f}")
print(f"Средний предсказанный PD: {p_test.mean():.4f}   |   Фактическая доля дефолтов: {y_test.mean():.4f}")

In [ ]:
pd.Series(model.coef_[0], index=keep_features).sort_values().plot(kind="barh", figsize=(7, 6))
plt.title("Коэффициенты логистической регрессии (WOE-признаки)")
plt.xlabel("Коэффициент")
plt.tight_layout()
plt.show()

## 5. Диагностика

In [ ]:
fpr, tpr, _ = roc_curve(y_test, p_test)
plt.figure()
plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC-кривая")
plt.legend()
plt.savefig("../reports/figures/roc_curve.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
calib_df = pd.DataFrame({"pd": p_test, "actual": y_test.values})
calib_df["bucket"] = pd.qcut(calib_df["pd"], q=5, duplicates="drop")
calib_summary = calib_df.groupby("bucket", observed=True).agg(
    mean_predicted=("pd", "mean"), actual_rate=("actual", "mean")
)

plt.figure()
plt.plot(calib_summary["mean_predicted"], calib_summary["actual_rate"], marker="o", label="Модель")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Идеальная калибровка")
plt.xlabel("Средний предсказанный PD")
plt.ylabel("Фактическая доля дефолтов")
plt.title("Калибровочная кривая")
plt.legend()
plt.savefig("../reports/figures/calibration_curve.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
corr = X_train_woe[keep_features].corr()
high_corr = corr.where(~np.eye(len(corr), dtype=bool)).abs().max().max()
print(f"Максимальная попарная корреляция среди WOE-признаков: {high_corr:.3f}")

## 6. Перевод в баллы скоркарты

Offset = 600, PDO = 20 (points to double the odds).

In [ ]:
OFFSET, PDO = 600, 20
scorecard_table = build_scorecard(model, woe_tables, keep_features, offset=OFFSET, pdo=PDO)
scorecard_table.sort_values(["feature", "points"])

## 7. Пример расчёта — три заявителя с разным уровнем риска

In [ ]:
sorted_by_pd = np.argsort(model.predict_proba(X_test_woe[keep_features])[:, 1])
low_idx, high_idx = sorted_by_pd[0], sorted_by_pd[-1]

examples = []
for label, idx in [("Низкий риск", low_idx), ("Средний риск (первая строка теста)", 0), ("Высокий риск", high_idx)]:
    binned = {col: test_binned_cols[col].iloc[idx] for col in keep_features}
    points = score_applicant(binned, scorecard_table)
    pred_pd = model.predict_proba(X_test_woe[keep_features].iloc[[idx]])[:, 1][0]
    examples.append({"профиль": label, "баллы": round(points, 2), "PD": round(pred_pd, 4)})

pd.DataFrame(examples)

## 8. Проверка инференса

Подтверждает, что путь скоринга из сырых входных данных (`score_new_applicant`, используется
в API) даёт тот же результат, что и расчёт напрямую из обучающего пайплайна.

In [ ]:
raw_applicant = X_test.iloc[low_idx][keep_features].to_dict()
result = score_new_applicant(raw_applicant, binning_artifacts, woe_tables, scorecard_table, model, keep_features)
print(result)

## 9. Сохранение артефактов

In [ ]:
import os

import joblib

os.makedirs("../artifacts", exist_ok=True)
joblib.dump(model, "../artifacts/model.pkl")
joblib.dump(woe_tables, "../artifacts/woe_tables.pkl")
joblib.dump(scorecard_table, "../artifacts/scorecard_table.pkl")
joblib.dump(keep_features, "../artifacts/keep_features.pkl")
joblib.dump(binning_artifacts, "../artifacts/binning_artifacts.pkl")
print("Артефакты сохранены в artifacts/")

## 10. Итоги

- **Gini 0.604** (бенчмарк ≳0.4), **KS 0.571**, калибровка в пределах 0.5 п.п. от фактической доли дефолтов
- `checking_status` — доминирующий признак (IV 0.61), подтверждено, что это подлинный сигнал
  для данного датасета, а не утечка целевой переменной
- Полное обсуждение, ограничения и экономическая интерпретация — в `reports/executive_summary.md`
- Сервис развёрнут через FastAPI (`api/main.py`) и Streamlit (`app/streamlit_app.py`)